In [2]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import json
import tarfile
import xgboost as xgb_lib
# Weather data library
from meteostat import Stations, Daily
from datetime import datetime
from pyathena import connect

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

import sagemaker
from sagemaker.inputs import TrainingInput
from sagemaker import image_uris
from sagemaker.serializers import CSVSerializer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,accuracy_score, precision_score,recall_score, f1_score)
                                

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


# Connect SageMaker to S3

In [3]:
#Create S3 client using boto3
#This allows the notebook to communicate with S3
s3 = boto3.client('s3')

#Define your S3 bucket name where your datasets are stored
bucket = "vegetation-risk-ml"

#list files inside bucket
response = s3.list_objects_v2(Bucket=bucket)

# Load ML dataset from S3

In [4]:
vegetation_risk_data = pd.read_parquet("s3://vegetation-risk-ml/final/ml_ready/vegetation_ml_dataset.parquet")

print(vegetation_risk_data.shape)
vegetation_risk_data.head()

(243820, 25)


,dia,ht,slope,regional_drybiot,lat,lon,countycd,fire_count,max_fire_size,avg_fire_size,...,avg_wind,weather_distance_miles,fuel_moisture_risk,fire_recurrence,fuel_load_density,wildfire_risk_score,trim_priority,log_fire_size,fire_month_sin,fire_month_cos
0,10.2,46.0,35.0,404.669221,32.677701,-116.727918,73,2,263.0,156.5,...,11.297753,15.744269,15.883586,1.098612,12.154104,0.476166,High,5.575949,-1.0,-1.836970e-16
1,5.2,21.0,0.0,73.837135,32.692853,-116.508686,73,1,25.0,25.0,...,11.362007,5.792634,21.553977,0.693147,8.995167,0.331367,Medium,3.258097,-0.5,-8.660254e-01
2,6.0,23.0,0.0,108.927059,32.692853,-116.508686,73,1,25.0,25.0,...,11.362007,5.792634,21.553977,0.693147,9.617999,0.343172,Medium,3.258097,-0.5,-8.660254e-01
3,3.3,23.0,0.0,27.219884,32.761664,-116.727392,73,1,16390.0,16390.0,...,11.297753,18.134465,15.883586,0.693147,7.633848,0.257738,Low,9.704488,-1.0,-1.836970e-16
4,5.0,14.0,35.0,52.318863,32.761664,-116.727392,73,1,16390.0,16390.0,...,11.297753,18.134465,15.883586,0.693147,8.206125,0.268584,Low,9.704488,-1.0,-1.836970e-16


# Select features for training

In [6]:
#Map trim_priority to numeric labels
mapping = {'Low': 0, 'Medium': 1, 'High': 2}
vegetation_risk_data['target'] = vegetation_risk_data['trim_priority'].map(mapping)
if vegetation_risk_data['target'].isnull().any():
    print("Some trim_priority values were not mapped and resulted in NaN in the target column.")
else:
    print("All trim_priority values successfully mapped to target labels.")


#Save lat/lon/county before split
meta = vegetation_risk_data[['lat', 'lon', 'countycd']].copy().reset_index(drop=True)

All trim_priority values successfully mapped to target labels.


In [7]:
# Select features and target variable
features=['dia', 'ht', 'slope', 'regional_drybiot', 'fire_recurrence', 'avg_temp', 'avg_rain', 'avg_wind', 'fuel_moisture_risk', 'log_fire_size','fire_month_sin', 'fire_month_cos']

X = vegetation_risk_data[features].reset_index(drop=True)
y = vegetation_risk_data["target"].reset_index(drop=True)


# Train Test Split

In [8]:
#Use a stratified 70,15,15 (training,testing,validation) split to preserve class distribution across all three priority levels
#Split into 70% train, 30% temp (which will be further split into val and test)
X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(X, y, X.index, test_size=0.30, random_state=42, stratify=y)

# Split temp into 15% val, 15% test
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(X_temp, y_temp, idx_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Reset indices to avoid misalignment
X_train = X_train.reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val   = y_val.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Validation shape:", X_val.shape)

Train shape: (170674, 12)
Test shape: (36573, 12)
Validation shape: (36573, 12)


# Save Splits to S3

In [9]:
#Save and upload all three splits
def save_to_s3(X, y, bucket, key):
    df = pd.concat([y.reset_index(drop=True), X.reset_index(drop=True)], axis=1)
    df.to_csv(f's3://{bucket}/{key}', header=False, index=False)

BUCKET = 'vegetation-risk-ml'
save_to_s3(X_train, y_train, BUCKET, 'ml/train/train.csv')
save_to_s3(X_val,   y_val,   BUCKET, 'ml/val/val.csv')
save_to_s3(X_test,  y_test,  BUCKET, 'ml/test/test.csv')